# 📈 股票深度分析工作台

支持：**A股**（akshare）、**港股/美股**（yfinance）

---
## 使用指南
1. 修改下方 `SYMBOL` 和 `MARKET` 变量
2. 依次运行各个 Cell
3. 根据需要调整参数

In [ ]:
# ========== 配置区 ==========
SYMBOL = "600519"        # 股票代码
MARKET = "a_share"       # a_share / us / hk
TIMEFRAME = "1y"          # 1mo / 3mo / 6mo / 1y / 2y / 5y / max

# A股: 600519(茅台) 000001(平安银行) 000858(五粮液) 300750(宁德时代)
# 港股: 00700(腾讯) 09988(阿里) 01810(小米)
# 美股: AAPL TSLA NVDA MSFT GOOGL META AMZN

In [ ]:
# ========== 导入库 ==========
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# 中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print(f"✅ 环境就绪 | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## 1️⃣ 实时行情

In [ ]:
def get_realtime(symbol, market):
    """获取实时行情"""
    if market == "a_share":
        import akshare as ak
        code = symbol.replace("sh","").replace("sz","")
        df = ak.stock_zh_a_spot_em()
        match = df[df["代码"] == code]
        if match.empty:
            print(f"❌ 未找到: {symbol}")
            return {}
        row = match.iloc[0]
        return {
            "名称": row.get("名称"), "代码": row.get("代码"),
            "最新价": row.get("最新价"), "昨收": row.get("昨收"),
            "涨跌额": row.get("涨跌额"), "涨跌幅": row.get("涨跌幅"),
            "最高": row.get("最高"), "最低": row.get("最低"),
            "今开": row.get("今开"), "成交量": row.get("成交量"),
            "成交额": row.get("成交额"), "换手率": row.get("换手率"),
            "市盈率": row.get("市盈率-动态"), "总市值": row.get("总市值"),
        }
    else:
        import yfinance as yf
        t = yf.Ticker(symbol)
        info = t.info
        price = info.get("currentPrice") or info.get("regularMarketPrice") or info.get("previousClose")
        prev = info.get("previousClose") or info.get("regularMarketPreviousClose")
        change = (price - prev) if price and prev else None
        change_pct = (change / prev * 100) if change and prev else None
        return {
            "名称": info.get("longName") or info.get("shortName", symbol),
            "代码": symbol,
            "最新价": price, "昨收": prev,
            "涨跌额": round(change, 2) if change else None,
            "涨跌幅": f"{change_pct:+.2f}%" if change_pct else None,
            "最高": info.get("dayHigh"), "最低": info.get("dayLow"),
            "今开": info.get("regularMarketOpen"),
            "成交量": info.get("regularMarketVolume") or info.get("volume"),
            "市值": info.get("marketCap"),
            "市盈率": info.get("trailingPE") or info.get("forwardPE"),
            "52周高": info.get("fiftyTwoWeekHigh"),
            "52周低": info.get("fiftyTwoWeekLow"),
            "货币": info.get("currency", "USD"),
        }

info = get_realtime(SYMBOL, MARKET)
if info:
    for k, v in info.items():
        if v is not None:
            print(f"  {k}: {v}")
else:
    print("⚠️ 未能获取实时数据（可能非交易时间）")

---
## 2️⃣ 历史K线数据

In [ ]:
def _add_a_prefix(code):
    """给A股代码加 sh/sz 前缀"""
    code = code.replace("sh","").replace("sz","").strip()
    if code.startswith(("6", "5", "9")):
        return f"sh{code}"
    else:
        return f"sz{code}"

def get_history(symbol, market, timeframe="1y"):
    """获取历史K线"""
    period_map = {"1mo":30,"3mo":90,"6mo":180,"1y":365,"2y":730,"5y":1825,"max":3650}
    cutoff = datetime.now() - timedelta(days=period_map.get(timeframe, 365))
    
    if market == "a_share":
        import akshare as ak
        prefixed = _add_a_prefix(symbol)
        # 优先新浪数据源（更稳定）
        try:
            df = ak.stock_zh_a_daily(symbol=prefixed, adjust="qfq")
            if df is not None and not df.empty:
                df = df.rename(columns={"date":"Date","open":"Open","close":"Close","high":"High","low":"Low","volume":"Volume","amount":"Amount","turnover":"Turnover"})
                df["Date"] = pd.to_datetime(df["Date"])
                df = df.set_index("Date").sort_index()
                df = df[df.index >= pd.Timestamp(cutoff)]
            return df
        except Exception as e1:
            # 备用：腾讯数据源
            try:
                end = datetime.now().strftime("%Y%m%d")
                start = cutoff.strftime("%Y%m%d")
                df = ak.stock_zh_a_hist_tx(symbol=prefixed, start_date=start, end_date=end)
                if df is not None and not df.empty:
                    df = df.rename(columns={"date":"Date","open":"Open","close":"Close","high":"High","low":"Low","amount":"Amount"})
                    df["Date"] = pd.to_datetime(df["Date"])
                    df = df.set_index("Date").sort_index()
                    if "Volume" not in df.columns:
                        df["Volume"] = 0
                return df
            except Exception as e2:
                print(f"⚠️ A股历史数据获取失败: {e1}; {e2}")
                return pd.DataFrame()
    else:
        import yfinance as yf
        interval_map = {"1mo":("1mo","60m"),"3mo":("3mo","1d"),"6mo":("6mo","1d"),"1y":("1y","1d"),"2y":("2y","1d"),"5y":("5y","1wk"),"max":("max","1wk")}
        yf_period, yf_interval = interval_map.get(timeframe, ("1y","1d"))
        try:
            t = yf.Ticker(symbol)
            return t.history(period=yf_period, interval=yf_interval)
        except Exception as e:
            print(f"⚠️ yfinance获取失败（可能被限流）: {e}")
            return pd.DataFrame()

df = get_history(SYMBOL, MARKET, TIMEFRAME)
if not df.empty:
    print(f"📊 {SYMBOL} | 数据点数: {len(df)} | {df.index[0].strftime('%Y-%m-%d')} → {df.index[-1].strftime('%Y-%m-%d')}")
    print(f"   期初: {df['Close'].iloc[0]:.2f} → 期末: {df['Close'].iloc[-1]:.2f}")
    print(f"   涨跌幅: {(df['Close'].iloc[-1]/df['Close'].iloc[0]-1)*100:+.2f}%")
    df.tail()
else:
    print("❌ 未能获取数据")

---
## 3️⃣ K线图（含均线 + 布林带 + MACD）

In [ ]:
# 计算技术指标
df["MA5"] = df["Close"].rolling(5).mean()
df["MA20"] = df["Close"].rolling(20).mean()
df["MA60"] = df["Close"].rolling(60).mean()
df["MA120"] = df["Close"].rolling(120).mean()

# 布林带 (20,2)
df["BB_Mid"] = df["Close"].rolling(20).mean()
df["BB_Std"] = df["Close"].rolling(20).std()
df["BB_Upper"] = df["BB_Mid"] + 2 * df["BB_Std"]
df["BB_Lower"] = df["BB_Mid"] - 2 * df["BB_Std"]

# MACD
ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()
df["MACD"] = ema12 - ema26
df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
df["MACD_Hist"] = df["MACD"] - df["MACD_Signal"]

# RSI (14)
delta = df["Close"].diff()
gain = delta.where(delta > 0, 0.0).rolling(14).mean()
loss = (-delta.where(delta < 0, 0.0)).rolling(14).mean()
rs = gain / loss
df["RSI"] = 100 - (100 / (1 + rs))

print("✅ 技术指标计算完成")
print(f"  MA20: {df['MA20'].iloc[-1]:.2f}")
print(f"  布林上轨: {df['BB_Upper'].iloc[-1]:.2f} / 下轨: {df['BB_Lower'].iloc[-1]:.2f}")
print(f"  MACD: {df['MACD'].iloc[-1]:.3f}")
print(f"  RSI(14): {df['RSI'].iloc[-1]:.1f}")

In [ ]:
# ========== 完整技术分析图 ==========
fig, axes = plt.subplots(4, 1, figsize=(16, 14), gridspec_kw={'height_ratios': [3, 1, 1, 1]})
ax1, ax2, ax3, ax4 = axes

# 1) K线 + 均线 + 布林带
colors = ['#ef5350' if df['Close'].iloc[i] >= df['Open'].iloc[i] else '#26a69a' for i in range(len(df))]
body_w = 0.6
for i, (idx, row) in enumerate(df.iterrows()):
    ax1.plot([idx, idx], [row['Low'], row['High']], color=colors[i], linewidth=0.5)
    bh = abs(row['Close'] - row['Open'])
    if bh > 0:
        ax1.bar(idx, bh, body_w, bottom=min(row['Open'], row['Close']), color=colors[i])

ax1.plot(df.index, df['MA20'], color='#64B5F6', linewidth=1.2, label='MA20')
ax1.plot(df.index, df['MA60'], color='#CE93D8', linewidth=1.2, label='MA60')
ax1.plot(df.index, df['BB_Upper'], color='gray', linewidth=0.7, linestyle='--', alpha=0.6, label='BB Upper/Lower')
ax1.plot(df.index, df['BB_Lower'], color='gray', linewidth=0.7, linestyle='--', alpha=0.6)
ax1.fill_between(df.index, df['BB_Upper'], df['BB_Lower'], alpha=0.05, color='gray')
ax1.set_title(f'{SYMBOL} 技术分析 | {TIMEFRAME}', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.set_ylabel('Price')

# 2) 成交量
ax2.bar(df.index, df['Volume'], color=colors, alpha=0.6, width=body_w)
ax2.set_ylabel('Volume')
ax2.grid(True, alpha=0.3)

# 3) MACD
macd_colors = ['#ef5350' if v >= 0 else '#26a69a' for v in df['MACD_Hist']]
ax3.bar(df.index, df['MACD_Hist'], color=macd_colors, width=body_w)
ax3.plot(df.index, df['MACD'], color='#1E88E5', linewidth=1.5, label='MACD')
ax3.plot(df.index, df['MACD_Signal'], color='#FF9800', linewidth=1, label='Signal')
ax3.axhline(y=0, color='gray', linewidth=0.5, linestyle='-')
ax3.set_ylabel('MACD')
ax3.legend(loc='upper left', fontsize=7)
ax3.grid(True, alpha=0.3)

# 4) RSI
ax4.plot(df.index, df['RSI'], color='#7E57C2', linewidth=1.2, label='RSI(14)')
ax4.axhline(y=70, color='#ef5350', linewidth=0.8, linestyle='--', alpha=0.7)
ax4.axhline(y=30, color='#26a69a', linewidth=0.8, linestyle='--', alpha=0.7)
ax4.fill_between(df.index, 70, 100, alpha=0.1, color='#ef5350')
ax4.fill_between(df.index, 0, 30, alpha=0.1, color='#26a69a')
ax4.set_ylim(0, 100)
ax4.set_ylabel('RSI')
ax4.legend(loc='upper left', fontsize=7)
ax4.grid(True, alpha=0.3)

# 日期格式
for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=max(1, len(df)//12)))

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

---
## 4️⃣ 收益率分析

In [ ]:
# 计算日收益率
df["Return"] = df["Close"].pct_change()

# 累计收益
df["CumReturn"] = (1 + df["Return"]).cumprod() - 1

# 统计摘要
annual_return = df["Return"].mean() * 252
annual_vol = df["Return"].std() * np.sqrt(252)
sharpe = annual_return / annual_vol if annual_vol > 0 else 0
max_drawdown = (df["CumReturn"] - df["CumReturn"].cummax()).min()

print("📊 收益风险指标")
print(f"  {'─'*40}")
print(f"  年化收益率: {annual_return*100:.2f}%")
print(f"  年化波动率: {annual_vol*100:.2f}%")
print(f"  Sharpe 比率: {sharpe:.2f}")
print(f"  最大回撤:   {max_drawdown*100:.2f}%")
print(f"  胜率(日):   {(df['Return'] > 0).mean()*100:.1f}%")
print(f"  日均收益:   {df['Return'].mean()*100:.3f}%")

# 月度收益热力图数据
monthly = df["Return"].resample("M").apply(lambda x: (1+x).prod()-1)
print(f"\n📅 月度收益 (最近12个月):")
for m, r in monthly.tail(12).items():
    tag = "🔴" if r > 0 else "🟢"
    print(f"  {m.strftime('%Y-%m')}: {tag} {r*100:+.2f}%")

---
## 5️⃣ 与大盘对比（A股 vs 沪深300）

In [ ]:
if MARKET == "a_share":
    # 获取沪深300基准
    import akshare as ak
    end = datetime.now().strftime("%Y%m%d")
    period_map = {"1mo":30,"3mo":90,"6mo":180,"1y":365,"2y":730,"5y":1825,"max":3650}
    start = (datetime.now() - timedelta(days=period_map.get(TIMEFRAME, 365))).strftime("%Y%m%d")
    
    try:
        hs300 = ak.stock_zh_index_daily(symbol="sh000300")
        hs300["date"] = pd.to_datetime(hs300["date"])
        hs300 = hs300.set_index("date").sort_index()
        hs300 = hs300.loc[start:end]
        
        # 归一化比较
        stock_norm = df["Close"] / df["Close"].iloc[0]
        hs300_norm = hs300["close"] / hs300["close"].iloc[0]
        
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.plot(stock_norm.index, stock_norm, color='#ef5350', linewidth=2, label=f'{SYMBOL}')
        ax.plot(hs300_norm.index, hs300_norm, color='#64B5F6', linewidth=2, label='沪深300')
        ax.axhline(y=1, color='gray', linewidth=0.5, linestyle='--')
        ax.fill_between(stock_norm.index, stock_norm, 1, 
                        where=(stock_norm >= 1), color='#ef5350', alpha=0.1)
        ax.fill_between(stock_norm.index, stock_norm, 1,
                        where=(stock_norm < 1), color='#26a69a', alpha=0.1)
        ax.legend(fontsize=11)
        ax.set_title(f'{SYMBOL} vs 沪深300 相对强弱 | {TIMEFRAME}', fontsize=14, fontweight='bold')
        ax.set_ylabel('归一化价格 (起点=1)')
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()
        
        # 超额收益
        excess = (stock_norm.iloc[-1] - hs300_norm.iloc[-1]) * 100
        print(f"\n📈 相对沪深300 超额收益: {excess:+.2f}%")
    except Exception as e:
        print(f"⚠️ 无法获取沪深300数据: {e}")
else:
    print("⚠️ 此功能仅支持A股（需要akshare获取沪深300指数）")

---
## 6️⃣ 自定义多股对比

In [ ]:
# 自定义对比列表
COMPARE_LIST = [
    # ("600519", "a_share", "茅台"),
    # ("000858", "a_share", "五粮液"),
]

if COMPARE_LIST:
    fig, ax = plt.subplots(figsize=(14, 6))
    colors = ['#ef5350', '#64B5F6', '#FF9800', '#4CAF50', '#9C27B0', '#00BCD4']
    
    for i, (sym, mkt, label) in enumerate(COMPARE_LIST):
        d = get_history(sym, mkt, TIMEFRAME)
        if not d.empty:
            norm = d["Close"] / d["Close"].iloc[0]
            ax.plot(norm.index, norm, color=colors[i % len(colors)], linewidth=2, label=f'{label}({sym})')
    
    ax.axhline(y=1, color='gray', linewidth=0.5, linestyle='--')
    ax.legend(fontsize=10)
    ax.set_title(f'多股对比 | {TIMEFRAME}', fontsize=14, fontweight='bold')
    ax.set_ylabel('归一化价格 (起点=1)')
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
else:
    print("💡 在上方 COMPARE_LIST 中添加股票代码即可对比")


---
### 📝 分析笔记

在此记录你的观察和判断：
- 趋势判断：
- 关键价位：
- 异常信号：
- 操作计划：